# narrative_dna Colab Quickstart

Este notebook descarga el repo desde GitHub, instala `narrative_dna`, importa los módulos principales y ejecuta un flujo JSON-first conservador. La notación compacta se deriva siempre desde JSON validado.

## 1. Clonar el repo desde GitHub

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/jcval94/ADNarrativa.git"
REPO_DIR = Path("/content/ADNarrativa")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(Path.cwd())

## 2. Instalar el paquete

In [ ]:
%pip install -q -e ".[dev]"

## 3. Importar módulos principales

In [ ]:
import json
from pathlib import Path

from narrative_dna.evaluator import load_gold_units
from narrative_dna.loader import load_documents
from narrative_dna.pipeline import run_pipeline

print("Imports OK")

## 4. Cargar una transcripción de ejemplo sin LLM

In [ ]:
documents = load_documents("data/transcripts/videos", limit=1)
document = documents[0]

print("document_id:", document.document_id)
print("units:", len(document.units))
print(document.units[0].model_dump(mode="json"))

## 5. Ejecutar el pipeline JSON-first sin LLM

Este modo es conservador: genera unidades finales `N_N0{0}` y agrega heurísticas sólo como señales candidatas auditables.

In [ ]:
result = run_pipeline(
    input_dir="data/transcripts/videos",
    output_dir="outputs",
    run_id="colab_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
    limit=1,
)

print("run_id:", result.run_id)
print("run_dir:", result.run_dir)
print("documents:", len(result.documents))

## 6. Leer outputs JSON/JSONL

In [ ]:
run_dir = Path("outputs/colab_no_llm_demo")

manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
print("manifest run_id:", manifest["run_id"])
print("taxonomy:", manifest["taxonomy_version"])

print("\nPrimeras unidades:")
for line in (run_dir / "units.jsonl").read_text(encoding="utf-8").splitlines()[:3]:
    unit = json.loads(line)
    print(unit["unit_id"], unit["final_notation"], unit["text"][:100])

print("\nAudit report preview:")
print((run_dir / "audit_report.md").read_text(encoding="utf-8")[:1000])

## 7. Probar regresión golden

In [ ]:
!python -m pytest tests/test_golden_regression.py -q

## 8. Alternativa por CLI

In [ ]:
!narrative-dna run --input-dir data/transcripts/videos --output-dir outputs --run-id colab_cli_no_llm --no-llm --no-adjudicator --limit 1
!narrative-dna inspect --run-id colab_cli_no_llm

## 9. Opcional: usar OpenAI desde Colab Secrets

En Colab, guarda `OPENAI_API_KEY` en Secrets. Después descomenta y ejecuta esta celda para clasificar con LLM y adjudicator conservador.

In [ ]:
# from google.colab import userdata
# import os
#
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
#
# llm_result = run_pipeline(
#     input_dir="data/transcripts/videos",
#     output_dir="outputs",
#     run_id="colab_llm_demo",
#     use_llm=True,
#     use_adjudicator=True,
#     audit_similarity_enabled=True,
#     limit=1,
# )
# print(llm_result.run_dir)

## 10. Evaluar contra synthetic gold high-confidence

Cuando tengas `outputs/<RUN_ID>/synthetic_gold_high_confidence.jsonl`, evalúa así:

In [ ]:
# !narrative-dna evaluate --run-id colab_llm_demo --gold outputs/colab_llm_demo/synthetic_gold_high_confidence.jsonl